In [ ]:
import os
import sys
import json
from pathlib import Path


import pandas as pd
import numpy as np
import altair as alt


import geopandas as gpd
from shapely.geometry import Point

pd.set_option('display.max_rows', None)

pd.set_option('display.max_columns', None)
pd.set_option("display.width", 160)

print("Versions ->",
      "pandas:", pd.__version__,
      "| geopandas:", gpd.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
file_path = '/content/drive/MyDrive/424-Assignment-3/df_sub.pkl'
DF_SUB = pd.read_pickle(file_path)

DF_SUB.dtypes

In [ ]:
FINAL_DF = DF_SUB

# Linked View Visualization 3: Dwelling Units — Adds vs Removals by Borough


In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

df0 = FINAL_DF[[
    "Filing Date", "Borough", "Job Type",
    "Existing Dwelling Units", "Proposed Dwelling Units"
]].dropna(subset=["Filing Date", "Borough"]).copy()

df0["Filing Date"] = pd.to_datetime(df0["Filing Date"], errors="coerce")
df0 = df0.dropna(subset=["Filing Date"])

df0["Existing Dwelling Units"] = pd.to_numeric(df0["Existing Dwelling Units"], errors="coerce")
df0["Proposed Dwelling Units"] = pd.to_numeric(df0["Proposed Dwelling Units"], errors="coerce")
df0 = df0.dropna(subset=["Existing Dwelling Units", "Proposed Dwelling Units"])

# Calendar fields
df0["year"]       = df0["Filing Date"].dt.year.astype(int)
df0["year_month"] = df0["Filing Date"].dt.to_period("M").dt.to_timestamp()

# Adds / removals
df0["delta_units"] = df0["Proposed Dwelling Units"] - df0["Existing Dwelling Units"]
df0["adds"]        = df0["delta_units"].clip(lower=0)
df0["removals"]    = (-df0["delta_units"].clip(upper=0))

years_available = sorted([y for y in df0["year"].unique().tolist() if 2021 <= y <= 2025]) or sorted(df0["year"].unique())
year_options    = years_available
jobtype_opts    = ["(All)"] + sorted(df0["Job Type"].dropna().astype(str).unique().tolist())

# Shared interactions
boroughSel = alt.selection_point(
    fields=["Borough"],
    name="boroughSel",
    bind="legend",
    toggle=True,
    empty=True
)

p_year = alt.param(
    name="yearSel",
    value=years_available[-1],
    bind=alt.binding_select(options=year_options, name="Year")
)
p_job = alt.param(
    name="jobSel",
    value="(All)",
    bind=alt.binding_select(options=jobtype_opts, name="Job Type")
)

# VIEW 1: Butterfly (linked by borough)
butter_base = (
    alt.Chart(df0)
      .transform_filter("(yearSel != 'All Years') ? datum.year == yearSel : true")
      .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
      .transform_filter("datum.passJob")
      .transform_calculate(
          adds_pos = "datum.delta_units > 0 ? datum.delta_units : 0",
          rem_pos  = "datum.delta_units < 0 ? -datum.delta_units : 0"
      )
      .transform_aggregate(
          adds="sum(adds_pos)", removals="sum(rem_pos)",
          groupby=["Borough"]
      )
      .transform_calculate(total_activity="datum.adds + datum.removals")
      .transform_fold(["adds","removals"], as_=["kind","value"])
      .transform_calculate(
          signed_value="datum.kind == 'adds' ? datum.value : -datum.value",
          kind_label="datum.kind == 'adds' ? 'Adds' : 'Removals'"
      )
)

zero_rule = alt.Chart(pd.DataFrame({"x":[0]})).mark_rule().encode(x="x:Q")

butter_bars = (
    butter_base
      .mark_bar()
      .add_params(boroughSel, p_year, p_job)
      .encode(
          y=alt.Y("Borough:N",
                  sort=alt.SortField(field="total_activity", order="descending"),
                  title=None),
          x=alt.X("signed_value:Q",
                  title="Dwelling Units (adds right, removals left)",
                  axis=alt.Axis(format="~s")),
          color=alt.Color("kind_label:N",
                          scale=alt.Scale(domain=["Adds","Removals"], range=["#4C78A8","#F28E2B"]),
                          legend=alt.Legend(title=None, orient="bottom")),
          opacity=alt.condition(boroughSel, alt.value(1), alt.value(0.35)),
          tooltip=[
              alt.Tooltip("Borough:N"),
              alt.Tooltip("kind_label:N", title="Kind"),
              alt.Tooltip("value:Q", title="Units", format=",.0f")
          ]
      )
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")  # show all if none selected
      .properties(width=820, height=240)
)

butter_labels = (
    butter_base
      .mark_text(align="left", dx=4, fontSize=11)
      .encode(
          y=alt.Y("Borough:N", sort=alt.SortField(field="total_activity", order="descending")),
          x="signed_value:Q",
          detail="kind_label:N",
          text=alt.Text("value:Q", format=",.0f"),
          opacity=alt.condition(boroughSel, alt.value(1), alt.value(0.35))
      )
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
)

butter_title = (
    alt.Chart(pd.DataFrame({"t":["Dwelling Units — Adds vs Removals by Borough (click legend/bars to select)"]}))
      .mark_text(align="left", fontSize=13, fontWeight="bold")
      .encode(text="t:N")
      .properties(height=28, width=820)
)

butterfly_view = alt.vconcat(
    butter_title,
    (zero_rule + butter_bars + butter_labels)
).resolve_scale(x="shared")

# VIEW 2: Monthly diverging clustered bars (linked)
# Monthly totals per Borough
monthly = (
    df0.groupby(["year","year_month","Borough","Job Type"], observed=False)[["adds","removals"]]
      .sum()
      .reset_index()
)

# Long + signed units (adds positive, removals negative)
monthly_long = monthly.melt(
    id_vars=["year","year_month","Borough","Job Type"],
    value_vars=["adds","removals"],
    var_name="kind",
    value_name="units"
)
monthly_long["kind_label"] = monthly_long["kind"].map({"adds":"Adds","removals":"Removals"})
monthly_long["signed_units"] = monthly_long.apply(
    lambda r: r["units"] if r["kind"]=="adds" else -r["units"], axis=1
)

monthly_chart = (
    alt.Chart(monthly_long, title="Monthly Adds (up) & Removals (down) — Diverging Clustered Bars")
      .transform_filter("(yearSel == 'All Years') || (datum.year == yearSel)")
      .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
      .transform_filter("datum.passJob")
      .add_params(boroughSel, p_year, p_job)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .mark_bar()
      .encode(
          x=alt.X("yearmonth(year_month):T", title="Month", axis=alt.Axis(format="%b")),
          y=alt.Y("signed_units:Q",
                  title="Dwelling Units (adds up, removals down)",
                  stack=None),
          xOffset=alt.XOffset("Borough:N"),
          color=alt.Color("Borough:N", scale=alt.Scale(scheme="category10")),
          opacity=alt.condition(boroughSel, alt.value(1.0), alt.value(0.25)),
          tooltip=[
              alt.Tooltip("yearmonth(year_month):T", title="Month", format="%B %Y"),
              alt.Tooltip("Borough:N"),
              alt.Tooltip("kind_label:N", title="Type"),
              alt.Tooltip("units:Q", title="Units", format=",.0f")
          ]
      )
      .properties(width=820, height=260)
)

# Zero baseline line for diverging reference
zero_hline = alt.Chart(pd.DataFrame({"y":[0]})).mark_rule(stroke="#888").encode(y="y:Q")

monthly_diverging_view = zero_hline + monthly_chart

# Final linked visualization
linked_viz = alt.vconcat(
    butterfly_view,
    monthly_diverging_view
).resolve_scale(
    color="independent"
).configure_axis(
    labelAngle=0, grid=True
).configure_view(
    stroke=None
)

linked_viz